In [3]:
#iris dataset ML pipeline

"""
Iris Dataset - ML Pipeline with Scaling, Encoding & Classification
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_curve, auc
)
from itertools import cycle

In [4]:
# ── 1. Load & Prepare Data ────────────────────────────────────────────────────
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y_raw = iris.target                          # numeric labels (0,1,2)
class_names = iris.target_names             # ['setosa','versicolor','virginica']

# LabelEncoder (demonstrates encoding step)
le = LabelEncoder()
le.fit(class_names)
y = le.transform(class_names[y_raw])        # same numeric values, encoder fitted

print("=" * 60)
print("  IRIS DATASET - ML PIPELINE")
print("=" * 60)
print(f"\nDataset shape : {X.shape}")
print(f"Classes       : {list(class_names)}")
print(f"Samples/class : {np.bincount(y_raw)}\n")

# ── 2. Train / Test Split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

##── 3. Build Pipelines ────────────────────────────────────────────────────────
pipelines = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=200, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    SVC(kernel="rbf", probability=True, random_state=42))
    ]),
}
# ── 4. Train & Evaluate ───────────────────────────────────────────────────────
results = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    cv     = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    results[name] = {
        "pipeline"  : pipe,
        "y_pred"    : y_pred,
        "accuracy"  : accuracy_score(y_test, y_pred),
        "cv_mean"   : cv.mean(),
        "cv_std"    : cv.std(),
        "report"    : classification_report(y_test, y_pred, target_names=class_names),
        "cm"        : confusion_matrix(y_test, y_pred),
    }
    print(f"[{name}]  Test Acc: {results[name]['accuracy']:.4f} | "
          f"CV: {cv.mean():.4f} ± {cv.std():.4f}")
 
best_name = max(results, key=lambda n: results[n]["accuracy"])
print(f"\n★ Best model: {best_name} ({results[best_name]['accuracy']:.4f})\n")
print(results[best_name]["report"])

  IRIS DATASET - ML PIPELINE

Dataset shape : (150, 4)
Classes       : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Samples/class : [50 50 50]

[Logistic Regression]  Test Acc: 0.9333 | CV: 0.9600 ± 0.0389
[Random Forest]  Test Acc: 0.9000 | CV: 0.9667 ± 0.0211
[SVM (RBF)]  Test Acc: 0.9667 | CV: 0.9667 ± 0.0211

★ Best model: SVM (RBF) (0.9667)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [5]:
import pickle

best_pipeline = results[best_name]["pipeline"]

with open("model.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print(f"Saved {best_name} pipeline as model.pkl")

Saved SVM (RBF) pipeline as model.pkl


In [ ]:
# Optional EDA summary that can run independently after the data cell.
if 'class_names' not in globals():
    class_names = np.array(getattr(iris, 'target_names', np.unique(y_raw).astype(str)))

print("\n-- Class Distribution --")
for i, name in enumerate(class_names):
    print(f"  {str(name):12s} : {np.sum(y_raw == i)} samples")
